# Experiment 4 — Deriving and solving deterministic collapse time

Run all cells in Colab (CPU). No probability or machine learning is used.

## 1. Equations and their solution

Let $A$ be the active node set, $n=|A|$, $L_i$ the load, $C_i$ capacity, and $D>0$ total demand per unit time. Between failures the model is

$$\frac{dL_i}{dt}=\frac{D}{n},\qquad L_i(t+\tau)=L_i(t)+\frac{D}{n}\tau.$$

Therefore the next capacity crossing occurs after

$$\tau_* = \frac{n}{D}\min_{i\in A}(C_i-L_i).$$

This is an exact solution between events, not a global collapse-time formula. At an event, remove the simultaneously failing nodes $F$, set $S=A\setminus F$, and redistribute synchronously:

$$L_j^+=L_j^-+\sum_{i\in F:\ j\in N(i)\cap S}\frac{L_i^-}{|N(i)\cap S|},\quad j\in S.$$

If the recipient set is empty, count that node's load as unserved demand. Repeat overload rounds at the same physical time. Then solve the next interval. Collapse time is the sum of these intervals until at least 30% of original nodes have failed.

**Boundary distinction:** in the continuous model, fail at upward capacity crossing (at equality). The existing discrete model fails only when load exceeds capacity after a whole demand step. These are different event conventions, not identical simulations.

For the original discrete model, the corresponding exact next-event step count is

$$k_* = \left\lfloor\frac{n\min_i(C_i-L_i)}{\Delta D}\right\rfloor+1.$$

This skips steps that cannot contain a failure. Floating-point boundary cases require checks. The continuous crossing time cannot simply be rounded to obtain whole-network discrete collapse time, because redistribution can change later events.

### Sources and scope

[Motter & Lai (2002)](https://arxiv.org/abs/cond-mat/0301086) motivates overload cascades, but their shortest-path load model differs from our accumulating neighbor-redistribution model. Continuous flows plus discrete jumps follow the general [hybrid-systems framework](https://webpages.math.luc.edu/~rgoebel1/publications/journals/SanfeliceGoebelTeel08ESAIMCOCV.pdf). The formulas above are derived here from our stated assumptions; we do not claim a new general theory.

Full-state, known-demand results are deterministic calculations. Future experiments can test simplified approximations or incomplete information, but re-solving the known model is not probabilistic forecasting.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML, display

SEED = 42
N = 100
MEAN_DEGREE = 6
CAPACITY = 10.0
INITIAL_UTILIZATION = (0.4, 0.7)
DEMAND_INCREMENT = 1.0
COLLAPSE_FRACTION = 0.30
MAX_STEPS = 1000

In [ ]:
def resolve_cascade(graph, loads, capacities, active, unserved=0.0):
    """Return a settled copy and a trace of synchronous failure rounds."""
    loads = dict(loads)
    active = set(active)
    trace = []
    while True:
        overloaded = {i for i in active if loads[i] > capacities[i]}
        if not overloaded:
            break
        survivors = active - overloaded
        additions = {i: 0.0 for i in survivors}
        for i in sorted(overloaded):
            recipients = sorted(set(graph.neighbors(i)) & survivors)
            if recipients:
                share = loads[i] / len(recipients)
                for j in recipients:
                    additions[j] += share
            else:
                unserved += loads[i]
            loads[i] = 0.0
        for j, amount in additions.items():
            loads[j] += amount
        active = survivors
        trace.append({'active': set(active), 'loads': dict(loads),
                      'failed': sorted(overloaded), 'unserved': unserved})
    return loads, active, unserved, trace

In [ ]:
def simulate(seed=SEED, n=N, mean_degree=MEAN_DEGREE,
             capacity=CAPACITY, utilization=INITIAL_UTILIZATION,
             delta_d=DEMAND_INCREMENT, collapse_fraction=COLLAPSE_FRACTION,
             max_steps=MAX_STEPS, capacity_spread=0.0):
    if not (n >= 2 and 0 < mean_degree <= n-1 and capacity > 0
            and 0 <= utilization[0] <= utilization[1] < 1
            and delta_d > 0 and 0 < collapse_fraction <= 1 and max_steps >= 1):
        raise ValueError('Invalid simulation settings')
    if n % 2 or not 0 <= capacity_spread <= 0.3:
        raise ValueError('This paired-capacity experiment requires even N and spread between 0 and 0.3')
    rng = np.random.default_rng(seed)
    for attempt in range(10000):
        graph = nx.gnp_random_graph(n, mean_degree/(n-1),
                                   seed=int(rng.integers(0, 2**32-1)))
        if nx.is_connected(graph):
            break
    else:
        raise RuntimeError('Could not sample connected graph; revise connectivity')
    capacities = dict.fromkeys(graph, float(capacity))
    loads = {i: capacity * rng.uniform(*utilization) for i in graph}
    # Separate RNG preserves identical graph and initial loads for each paired seed.
    capacity_rng = np.random.default_rng(np.random.SeedSequence([seed, 2026]))
    offsets = capacity_rng.uniform(0, 1, n//2)
    offsets = np.concatenate([offsets, -offsets])
    capacity_rng.shuffle(offsets)
    capacities = {i: capacity*(1+capacity_spread*offsets[i]) for i in graph}
    assert np.isclose(sum(capacities.values()), n*capacity)
    assert all(loads[i] < capacities[i] for i in graph)
    active = set(graph)
    initial = sum(loads.values())
    unserved = injected = 0.0
    history, frames = [], []
    collapse_step = None

    def frame(step, phase, failed=()):
        frames.append(dict(step=step, phase=phase, active=set(active),
                           loads=dict(loads), failed=list(failed)))

    def record(step, cascade_size):
        active_total = sum(loads[i] for i in active)
        assert np.isclose(active_total + unserved, initial + injected)
        spare = sum(capacities[i] - loads[i] for i in active)
        near = sum(loads[i]/capacities[i] >= .9 for i in active)
        giant = max((len(c) for c in nx.connected_components(graph.subgraph(active))), default=0)
        history.append(dict(step=step, injected=injected, failed=n-len(active),
                            failed_fraction=1-len(active)/n, spare=spare/(n*capacity),
                            near_fraction=near/len(active) if active else np.nan,
                            unserved=unserved, active_load=active_total,
                            cascade_size=cascade_size, giant_fraction=giant/n))

    record(0, 0)
    frame(0, 'initial')
    for step in range(1, max_steps+1):
        before = len(active)
        for i in active:
            loads[i] += delta_d / before
        injected += delta_d
        frame(step, 'demand added')
        loads, active, unserved, trace = resolve_cascade(
            graph, loads, capacities, active, unserved)
        for round_number, state in enumerate(trace, 1):
            assert np.isclose(sum(state['loads'][i] for i in state['active'])
                              + state['unserved'], initial + injected)
            frames.append(dict(step=step, phase=f'cascade round {round_number}',
                               active=state['active'], loads=state['loads'],
                               failed=state['failed']))
        record(step, before-len(active))
        if (n-len(active)) >= collapse_fraction*n - 1e-9:
            collapse_step = step
            break
    return dict(graph=graph, capacities=capacities, history=history,
                frames=frames, collapse_step=collapse_step,
                initial_load=initial, attempts=attempt+1)


In [ ]:
def solve_events(graph, initial_loads, capacities, demand=1.0, q=.3,
                 mode='discrete', max_time=1000):
    if mode not in ('discrete', 'continuous') or demand <= 0:
        raise ValueError('Invalid solver settings')
    loads = dict(initial_loads)
    active = set(graph)
    total_initial = sum(loads.values())
    lost = time = 0.0
    trace = []
    # A small tolerance is used only for continuous threshold events.
    tol = 1e-10
    for event in range(len(graph)):
        count = len(active)
        gap = min(capacities[i]-loads[i] for i in active)
        tau = count*max(0., gap)/demand
        wait = int(np.floor(tau))+1 if mode=='discrete' else tau
        if time+wait > max_time:
            return None, trace
        for i in active:
            loads[i] += demand*wait/count
        time += wait
        failed_this_event = 0
        while True:
            failing = {i for i in active if
                       (loads[i] > capacities[i] if mode=='discrete'
                        else loads[i] >= capacities[i]-tol)}
            if not failing:
                break
            survivors = active-failing
            additions = dict.fromkeys(survivors, 0.)
            for i in sorted(failing):
                neighbors = sorted(set(graph.neighbors(i)) & survivors)
                if neighbors:
                    for j in neighbors:
                        additions[j] += loads[i]/len(neighbors)
                else:
                    lost += loads[i]
                loads[i] = 0.
            for j in survivors:
                loads[j] += additions[j]
            failed_this_event += len(failing)
            active = survivors
        assert np.isclose(sum(loads[i] for i in active)+lost,
                          total_initial+demand*time)
        trace.append(dict(time=time, new_failures=failed_this_event,
                          total_failed=len(graph)-len(active)))
        if len(graph)-len(active) >= q*len(graph)-1e-9:
            return time, trace
        # A floating-point boundary may consume a discrete step without a failure.
        if not failed_this_event and mode=='continuous':
            raise RuntimeError('Continuous event failed to trigger')
    raise RuntimeError('Event limit reached; inspect floating-point boundaries')

# Generate the same initial conditions as Experiment 3.
# The solver receives ONLY graph, capacities, and initial loads.
reference = simulate(seed=1000, capacity_spread=.3)
initial = reference['frames'][0]['loads']
discrete_time, discrete_events = solve_events(reference['graph'], initial, reference['capacities'])
continuous_time, continuous_events = solve_events(reference['graph'], initial, reference['capacities'], mode='continuous')
print('Step-by-step discrete collapse:', reference['collapse_step'])
print('Event-skip discrete collapse:', discrete_time)
print('Continuous-time collapse:', round(continuous_time, 6))
assert discrete_time == reference['collapse_step']
print('Continuous event trace:')
for row in continuous_events:
    print({**row, 'time': round(row['time'], 6)})

## 2. Validate rather than assume equivalence

Compare the derived discrete event solver with the original step-by-step simulator over 150 paired cases. Small round-off differences are not silently accepted: every collapse time must match. The continuous model is reported separately because its failure timing convention differs.

In [ ]:
checks = []
for spread in (0., .2, .3):
    for seed in range(1000, 1050):
        baseline = simulate(seed=seed, capacity_spread=spread)
        calculated, events = solve_events(baseline['graph'], baseline['frames'][0]['loads'],
                                          baseline['capacities'])
        assert calculated == baseline['collapse_step'], (seed, spread, calculated, baseline['collapse_step'])
        checks.append((seed, spread, calculated, len(events)))
print(f'Passed {len(checks)} discrete collapse-time comparisons.')
print('Mean demand steps to collapse:', round(np.mean([r[2] for r in checks]), 2))
print('Mean event advances:', round(np.mean([r[3] for r in checks]), 2))
print('These operation counts are not a wall-clock speedup benchmark.')

In [ ]:
with plt.style.context('seaborn-v0_8-whitegrid'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for events, label in ((discrete_events, 'Discrete: whole demand steps'),
                           (continuous_events, 'Continuous: capacity crossings')):
        axes[0].step([0]+[r['time'] for r in events],
                     [0]+[r['total_failed'] for r in events], where='post', label=label)
    axes[0].axhline(COLLAPSE_FRACTION*N, color='gray', linestyle=':', label='Collapse threshold')
    axes[0].set_xlabel('Time (1 unit = one demand step duration)')
    axes[0].set_ylabel('Failed nodes')
    axes[0].legend(fontsize=8)
    axes[0].set_title('Two event conventions, same initial state')
    x = np.array([r[2] for r in checks])
    axes[1].scatter(x, x, s=12, alpha=.3)
    axes[1].plot([x.min(), x.max()], [x.min(), x.max()], 'k--')
    axes[1].set_xlabel('Step-by-step collapse time')
    axes[1].set_ylabel('Derived event-solver collapse time')
    axes[1].set_title('Validation: 150 matched cases')
    fig.tight_layout()
    plt.show()

## Parameters and limitations

Model parameters: N, graph connectivity, capacities and their spread, initial loads, demand rate, and collapse fraction. Numerical settings: the continuous event tolerance and maximum observation time. There is no trained model and no machine-learning hyperparameter.

The current validation uses random networks only. Comparing small-world and scale-free graphs remains necessary for the original topology-generalization objective. This notebook solves the known model; it does not establish accurate forecasts when demand, capacities, or observations are uncertain. A steady demand ramp alone can explain much of the baseline behavior. No equilibrium bifurcation or critical slowing down is claimed.